In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
import pandas as pd

In [ ]:
# enter pathlib path (dp) to cleaned notebook
dp = Path()

In [ ]:
data = pd.read_excel(dp)
pd.set_option("display.max_rows", 20)

#data["Datum"] = pd.to_datetime(data["Datum"], format="%Y%m%d")
data["datetime"] = pd.to_datetime(
    data["Datum"].astype(str) + " " + data["Tid"].astype(str),
    format="%Y%m%d %H:%M:%S"
)
cols = ["datetime"] + [c for c in data.columns if c != "datetime"]
data = data[cols]
data = data.sort_values("datetime")


columns_to_drop = ["Modalitet.1", "Efternamn", "Mellannamn", "Indikation", "Datum", "Tid", "Kommentar", "Användare", "Modalitetens serienummer", "Gravid", "Datum för föregående röntgenundersökning", "Extrafilter", "Grader", "Exp. Tid", "Röntgenrum", "Exp.läge", "Röntgenhuvud serienummer", "Röntgenrör serienummer", "Röntgenrörsmodell"]
data = data.drop(columns=columns_to_drop)

data["Födelsedatum"] = pd.to_datetime(
    data["Id"]
        .astype(str)
        .str.slice(0, 8),
    format="%Y%m%d",
    errors="coerce"
)

data["Ålder"] = (
    (data["datetime"] - data["Födelsedatum"])
    .dt.days // 365
)

mask = data["Kön"] == "O"

digit = pd.to_numeric(
    data.loc[mask, "Id"].astype(str).str[-2],
    errors="coerce"
)
data.loc[mask, "Kön"] = digit.mod(2).map({0: "F", 1: "M"})

data = data.drop(columns=("Id"))


data = data[data["Modalitet"] == 'CT']

pd.set_option("display.max_rows", 20)

data



In [ ]:
pd.set_option("display.max_rows", None)

# Antal US
for sex in ['F', 'M']:
    for age_range in ((16, 40), (41, 65), (66, 999), (0, 15)):

        a = data[
            data["datetime"].between(
                "2025-01-01",
                "2025-12-31 23:59:59"
            ) &
            (data["Modalitet"] == "CT") &
            (data["Kön"] == sex) &
            (data["Tänder"].isin(["Veraview X800", "3D Accuitomo 170"])) &
            (data["Ålder"].between(age_range[0], age_range[1]))
            ]
        print(f'antal {sex} {age_range[0]}-{age_range[1]}: {a.__len__()}')



In [ ]:
# DAP
age_ranges = ((16, 999), (0, 15))
for sex in ('F', 'M'):
    for age_range in age_ranges:
        a = data[
            data["datetime"].between(
                "2025-01-01",
                "2025-12-31 23:59:59"
            ) &
            (data["Modalitet"] == "CT") &
            (data["Kön"] == sex) &
            (data["Tänder"].isin(["Veraview X800", "3D Accuitomo 170"])) &
            (data["Ålder"].between(age_range[0], age_range[1]))
            ]

        dap = np.array(a["DAP (mGycm2)"]) * 1e-3 # DAP i Gycm2
        dap_mean = np.mean(dap)
        dap_median = np.median(dap)
        dap_q1, dap_q3 = np.percentile(dap, [25, 75])

        print(f'DAP mean {sex} ({age_range}): {dap_mean}')
        print(f'DAP median {sex} ({age_range}): {dap_median}')
        print(f'DAP Q1 {sex} ({age_range}): {dap_q1}')
        print(f'DAP Q3 {sex} ({age_range}): {dap_q3}')

        fig, ax = plt.subplots()
        ax.set_title(f'sex: {sex}, age range: {age_range}')
        ax.plot(dap, '.', label=f'DAP')
        ax.plot((0, a.__len__()), 2*[dap_mean], label='DAP mean')
        ax.plot((0, a.__len__()), 2*[dap_median], label='DAP median')
        ax.plot((0, a.__len__()), 2*[dap_q1], label='DAP Q1')
        ax.plot((0, a.__len__()), 2*[dap_q3], label='DAP Q3')
        ax.legend()
